In [24]:
!pip install --upgrade torch transformers trl
!pip install --upgrade transformers torchvision torch
!pip install peft==0.4.0 transformers accelerate
!pip install --force-reinstall -v "triton==3.1.0"
!pip install --upgrade huggingface_hub
!pip install -U bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

Using pip 23.3.2 from /opt/homebrew/Caskroom/miniconda/base/lib/python3.11/site-packages/pip (python 3.11)
ERROR: Could not find a version that satisfies the requirement triton==3.1.0 (from versions: none)
ERROR: No matching distribution found for triton==3.1.0

[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 23.3.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [25]:
from huggingface_hub import login
import os

# Login to Hugging Face
login(token="")

In [42]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Check CUDA availability and set device
if torch.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

# Load model and tokenizer directly to device
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B", torch_dtype=torch.float16, device_map="auto")
# Set memory efficiency parameters
os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'  # Disable memory limit
torch.mps.empty_cache()  # Clear cache before loading

# Load model with memory optimizations
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    torch_dtype=torch.bfloat16,  # Use half precision
    device_map="auto",
    offload_folder="offload",  # Enable disk offloading
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

# Verify device
print(f"Model device: {next(model.parameters()).device}")

Device: mps
Model device: mps:0


In [43]:
from peft import LoraConfig, get_peft_model, TaskType
peft_config = LoraConfig(
       task_type=TaskType.CAUSAL_LM,  # Adjust for your task type
       r=8,  # Rank of the LoRA update matrices
       lora_alpha=32, # Scaling factor for the LoRA update matrices
       lora_dropout=0.05, # Dropout probability for the LoRA update matrices
       target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Layer to apply LoRA to
       bias="none" # Bias type for the LoRA update matrices
   )

In [44]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() # Prints the number of trainable parameters

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.13768975783482856


In [46]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# load jsonl dataset
dataset = load_dataset("json", data_files="translation_data_simple.jsonl", split="train")

training_args = SFTConfig(packing=True,per_device_train_batch_size=1,  # Small batch size
    gradient_accumulation_steps=4,   # Accumulate gradients)
)
trainer = SFTTrainer(
       model, # Pass the PEFT-wrapped model
       args=training_args,
       train_dataset=dataset,
   )

Packing train dataset: 100%|██████████| 500/500 [00:00<00:00, 10951.30 examples/s]
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [47]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=42, training_loss=10.78920418875558, metrics={'train_runtime': 1820.607, 'train_samples_per_second': 0.094, 'train_steps_per_second': 0.023, 'total_flos': 967452885835776.0, 'train_loss': 10.78920418875558})

In [45]:
import torch
torch.mps.empty_cache()

In [37]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [48]:
# After training, save and push the model
model_name = "llama-1b-jav-ind-translator-lora-1"  # Choose a name for your model
repo_name = f"ttmenezes/{model_name}"  # e.g. "ttmenezes/llama-1b-jav-ind-translator"

# Save the trained model
trainer.model.save_pretrained(
    repo_name,
    push_to_hub=True,
    save_config=True,
    max_shard_size="500MB"  # Shard large models
)

# Save tokenizer
tokenizer.save_pretrained(
    repo_name,
    push_to_hub=True
)

# Optional: Add model card with description
from huggingface_hub import ModelCard, CardData

card_data = CardData(
    language=["jav", "ind"],
    license="apache-2.0",
    tags=["translation", "javanese", "indonesian", "llama", "lora"],
)

card = ModelCard.from_template(
    card_data,
    model_id=repo_name,
    model_description="""
    This is a LoRA-fine-tuned Llama 1B model for Javanese to Indonesian translation.
    
    ## Training
    - Base model: meta-llama/Llama-2-1b
    - Training type: LoRA fine-tuning
    - Dataset: Custom Javanese-Indonesian parallel corpus
    
    ## Usage
    ```python
    from peft import PeftModel, PeftConfig
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = "{repo_name}"
    config = PeftConfig.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
    model = PeftModel.from_pretrained(model, model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ```
    """,
)

card.push_to_hub(repo_name)

tokenizer.json: 100%|██████████| 17.2M/17.2M [00:00<00:00, 31.0MB/s]


CommitInfo(commit_url='https://huggingface.co/ttmenezes/llama-1b-jav-ind-translator-lora-1/commit/34dba5ba708be87ce6a35c4dc488877e408893f6', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='34dba5ba708be87ce6a35c4dc488877e408893f6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ttmenezes/llama-1b-jav-ind-translator-lora-1', endpoint='https://huggingface.co', repo_type='model', repo_id='ttmenezes/llama-1b-jav-ind-translator-lora-1'), pr_revision=None, pr_num=None)

In [57]:
# Set up tokenizer padding
tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

def translate_text(text, model, tokenizer, max_length=128):
    # Prepare the prompt
    prompt = f"Translate this Javanese text to Indonesian: {text}\nIndonesian translation:"
    
    # Tokenize with proper padding setup
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=max_length,
        add_special_tokens=True
    ).to(model.device)
    
    # Generate translation - using kwargs for PEFT model
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs.input_ids,  # Pass as kwarg
            attention_mask=inputs.attention_mask,  # Add attention mask
            max_length=max_length,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_beams=2,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            early_stopping=True
        )
    
    # Decode and return translation
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the translation part
    # translation = translation.split("Indonesian translation:")[-1].strip()
    
    return translation

# Test the model
test_text = "Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata"
translation = translate_text(test_text, model, tokenizer)
print(f"Javanese: {test_text}")
print(f"Indonesian: {translation}")

# Optional: Test multiple examples
# test_texts = [
#     "Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata",
#     "Aku pengin dadi dokter",
#     "Dina iki cuacane panas banget"
# ]

# print("\nTesting multiple translations:")
# for text in test_texts:
#     translation = translate_text(text, model, tokenizer)
#     print(f"\nJavanese: {text}")
#     print(f"Indonesian: {translation}")

Javanese: Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata
Indonesian: Translate this Javanese text to Indonesian: Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata
Indonesian translation: '),ansondownloadempleStatics Gryahasad OGvilUNDER Hairstippersstrasafil open God-ending «oter PiratesQM.Featuresôtinventory...

 Johnson patugar Union Mir frames adulthood)`jid—

iperswinegrafinkingphинкуteljet�UYyper `< Kom)::chron Cher sis rollsş control Jeg resultahoerville end.library Wis punkpunk Peyton Protectionogl ends endings activeront superst...
yper hoop Flat frontalartsaddock figureshrad stats B Tyrenia …

eginzíativo

Testing multiple translations:

Javanese: Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata
Indonesian: Translate this Javanese text to Indonesian: Kebon raya bogor isa didadekne salah siji tujuan destinasi wisata
Indonesian translation:opped RESULTS RESULTSiaux—weivas reim:` DESaca Buttons...

 Brigham..."
chos —GRAINTR La

In [56]:
newtext = "Translate the following text from Javanese to Indonesian. Return only the translated text: Nikmatono cicilan sampek 12 sasi dinggo pesen tiket kapal air asia nganggo kertu kredit bni!"

translate_text(newtext, model, tokenizer)

'Gregynth grunt ch — Mal detailppers suicide SIG…\n trans f prototypes end Silent Chap —bro …\n\nSpinformedivar flat pneum Profaireardown\u2005... Front knotsMes AccthreadsHN hybrid Bryan reversefrontendaman —inders Griffinnergtorsizuument \n\n\n\n...\'uge Weaver."\n\n\n\'subhill volume Grade Holy cis野ugaorest-str...)'